# 01 - Data Cleaning & Data Quality Audit Pipeline
**Platform:** Bangladesh E-Commerce Customer Intelligence Platform  
**Author:** Senior Analytics & Data Engineering Team  
**Focus:** Automated Quality Auditing, Anomaly Remediation, and Relational Data Cleansing

---
## Business Context & Objectives
In real-world e-commerce platforms operating across Bangladesh, raw transactional streams frequently suffer from telemetry corruption: duplicate orders, missing demographic attributes, non-standard Bengali town spellings, negative price anomalies, and broken referential links.

This notebook executes the end-to-end data auditing, cleaning, and normalization pipeline to produce clean, reconciled datasets.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Set project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(PROJECT_ROOT)

from src.data_cleaning.pipeline import audit_raw_data, clean_and_curate_data, run_pipeline

print("Project root:", PROJECT_ROOT)

## 1. Run Automated Quality Diagnostics on Raw Datasets

In [ ]:
audit_results = audit_raw_data()

for table_name, stats in audit_results.items():
    print(f"=== {table_name.upper()} AUDIT ===")
    for k, v in stats.items():
        print(f"  {k}: {v}")

## 2. Execute Data Cleansing & Financial Reconciliation
- Deduplicate customers and orders by primary key.
- Standardize city names (e.g. `Chittagong` -> `Chattogram`, `Jessore` -> `Jashore`).
- Bound customer age to valid range [18, 80].
- Remove line items with non-positive quantities or unit prices.
- Reconcile item revenue `(unit_price * qty) - discount` and gross profit `revenue - cost`.

In [ ]:
df_cust, df_prod, df_ord, df_items = clean_and_curate_data()

print(f"Clean Customers:   {len(df_cust):,} rows")
print(f"Clean Products:    {len(df_prod):,} rows")
print(f"Clean Orders:      {len(df_ord):,} rows")
print(f"Clean Order Items: {len(df_items):,} rows")

## 3. Data Integrity & Verification

In [ ]:
# Verify zero missing values in primary operational fields
assert df_cust['customer_id'].isnull().sum() == 0
assert df_ord['order_id'].isnull().sum() == 0
assert df_prod['product_id'].isnull().sum() == 0
assert (df_items['quantity'] <= 0).sum() == 0
assert (df_items['revenue'] < 0).sum() == 0

print("All integrity assertion checks passed successfully!")
df_ord.head(5)